In [10]:
import pandas as pd # veri okuma, isleme
import nltk # metin isleme
import re #regex ile metin temizleme
from nltk.corpus import stopwords # anlamsız kelimeri çıkarmak için
from nltk.stem import WordNetLemmatizer # lemma bulma işlemi

from sklearn.model_selection import train_test_split # eğitim/test bölme
from sklearn. feature_extraction.text import CountVectorizer # bag-of-words
from sklearn.tree import DecisionTreeClassifier #sınıflandırıcı model
from sklearn.metrics import confusion_matrix #başarı ölçümü

In [11]:
df=pd.read_csv("sms_spam.csv", encoding="latin-1")
print(df.head())

   type                                               text
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


In [12]:
#kolon isimlerini değiştir
df.columns=["label","text"]

#eksik değer var mı
print(f"Eksik değer sayıları: \n{df.isna().sum()}")

Eksik değer sayıları: 
label    0
text     0
dtype: int64


In [13]:
#metin ön işleme(temizlik, lower, stopwords, lemmatization)
nltk.download("stopwords")
nltk.download("wordnet") #lemmatizer için
nltk.download("omw-1.4") #farklı dil deteği

#lemmatizer oluştur
lemmatizer=WordNetLemmatizer()

#temizlenmiş verilerin listesi
clean_texts=[]

for msg in df["text"]:
  temp = re.sub("[^a-zA-Z]", " ", msg) # harf olmayan karakterleri çıkart
  temp = temp.lower() # tüm harfleri küçük hale getir
  temp = temp.split() # kelimeleri ayır

  #stopwords'leri çıkart
  temp=[word for word in temp if word not in stopwords.words("english")]

  #lemma
  temp=[lemmatizer.lemmatize(word) for word in temp]

  #kelimeri tekrar birleştir
  temp=" ".join(temp)

  clean_texts.append(temp)

df["clean_text"]=clean_texts
print(df.head())

[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


  label  ...                                         clean_text
0   ham  ...  go jurong point crazy available bugis n great ...
1   ham  ...                            ok lar joking wif u oni
2  spam  ...  free entry wkly comp win fa cup final tkts st ...
3   ham  ...                u dun say early hor u c already say
4   ham  ...                nah think go usf life around though

[5 rows x 3 columns]


In [14]:
#eğitim ve test veri seti
X = df["clean_text"] # bağımsız değişkenler
y = df["label"] # hedef değişken (spam/ham)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print (f"X_train shape: {X_train.shape}")
print (f"X_test shape: {X_test.shape}")

X_train shape: (4459,)
X_test shape: (1115,)


In [15]:
#feature extraction (bag of words)
cv = CountVectorizer()
X_train_cv = cv.fit_transform(X_train) # eğitim verisini dönüştür
X_test_cv = cv.transform(X_test) # test veri setini dönüştür


# model eğitimi
dt_model = DecisionTreeClassifier(random_state=42) # karar ağacı sınıflandır
dt_model.fit(X_train_cv, y_train) # training


# model degerlendirme
y_pred = dt_model.predict(X_test_cv) # test veri seti üzerinden tahmin yap


# confusion matrix hesapla
conf_matrix = confusion_matrix(y_test, y_pred)
print(conf_matrix)

# dogruluk
accuracy = 100 * (conf_matrix.trace() / conf_matrix.sum())
print (accuracy)

[[931  23]
 [ 23 138]]
95.87443946188341
